# Netflix Prize Recommender Without Spark

This notebook uses pandas and NumPy only. There is no Spark, PySpark, JVM, or Java dependency.

For a real Netflix Prize evaluation, we use the official `probe.txt` split. The final qualifying set does not include public labels, but the probe set does: `probe.txt` lists movie/customer pairs whose true ratings are present in the official `combined_data_*.txt` files. We remove those probe rows from training and evaluate RMSE on them.

## Expected Data

Place the original Netflix Prize files in `data/netflix/` or `../data/netflix/`:

- `combined_data_1.txt`
- `combined_data_2.txt`
- `combined_data_3.txt`
- `combined_data_4.txt`
- `probe.txt`

The parser below streams the raw challenge format directly, so no Spark preprocessing step is needed.

In [1]:
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd

DATA_DIR_CANDIDATES = [Path("../data/netflix"), Path("data/netflix")]
DATA_DIR = next((path for path in DATA_DIR_CANDIDATES if path.exists()), DATA_DIR_CANDIDATES[0])

COMBINED_FILES = [DATA_DIR / f"combined_data_{part}.txt" for part in range(1, 5)]
PROBE_FILE = DATA_DIR / "probe.txt"

RANDOM_SEED = 333
RNG = np.random.default_rng(RANDOM_SEED)

print(f"Using data directory: {DATA_DIR}")

Using data directory: ../data/netflix


In [2]:
missing = [path for path in [*COMBINED_FILES, PROBE_FILE] if not path.exists()]

if missing:
    missing_text = "\n".join(f"- {path}" for path in missing)
    raise FileNotFoundError(
        "Missing Netflix Prize files. Add the official challenge files before running:\n"
        f"{missing_text}"
    )

print("Found all required Netflix Prize files.")

Found all required Netflix Prize files.


## Load the Official Probe Split

The Netflix files use movie headers such as `123:`. `combined_data_*.txt` then has `customer_id,rating,date`; `probe.txt` has only customer IDs under each movie header.

In [3]:
def read_probe(path):
    rows = []
    movie_id = None

    with path.open("rt", encoding="utf-8") as file:
        for raw_line in file:
            line = raw_line.strip()
            if not line:
                continue

            if line.endswith(":"):
                movie_id = int(line[:-1])
            else:
                rows.append((movie_id, int(line)))

    probe = pd.DataFrame(rows, columns=["movie_id", "customer_id"])
    probe["movie_id"] = probe["movie_id"].astype("int32")
    probe["customer_id"] = probe["customer_id"].astype("int32")
    return probe


def iter_combined_files(paths):
    for path in paths:
        movie_id = None
        with path.open("rt", encoding="utf-8") as file:
            for raw_line in file:
                line = raw_line.strip()
                if not line:
                    continue

                if line.endswith(":"):
                    movie_id = int(line[:-1])
                else:
                    customer_text, rating_text, _date_text = line.split(",")
                    yield movie_id, int(customer_text), np.float32(rating_text)


probe_pairs = read_probe(PROBE_FILE)
probe_key_set = set(zip(probe_pairs["movie_id"].to_numpy(), probe_pairs["customer_id"].to_numpy()))

print(f"Probe rows: {len(probe_pairs):,}")
probe_pairs.head()

Probe rows: 1,408,395


,movie_id,customer_id
0,1,30878
1,1,2647871
2,1,1283744
3,1,2488120
4,1,317050


In [4]:
def load_train_and_probe_ratings(paths, probe_keys, max_train_rows=None):
    train_movies = []
    train_customers = []
    train_ratings = []
    probe_movies = []
    probe_customers = []
    probe_ratings = []

    started = perf_counter()

    for movie_id, customer_id, rating in iter_combined_files(paths):
        key = (movie_id, customer_id)

        if key in probe_keys:
            probe_movies.append(movie_id)
            probe_customers.append(customer_id)
            probe_ratings.append(rating)
            continue

        if max_train_rows is None or len(train_ratings) < max_train_rows:
            train_movies.append(movie_id)
            train_customers.append(customer_id)
            train_ratings.append(rating)

    train = pd.DataFrame({
        "movie_id": np.asarray(train_movies, dtype=np.int32),
        "customer_id": np.asarray(train_customers, dtype=np.int32),
        "rating": np.asarray(train_ratings, dtype=np.float32),
    })
    probe = pd.DataFrame({
        "movie_id": np.asarray(probe_movies, dtype=np.int32),
        "customer_id": np.asarray(probe_customers, dtype=np.int32),
        "rating": np.asarray(probe_ratings, dtype=np.float32),
    })

    elapsed = perf_counter() - started
    return train, probe, elapsed


# Leave as None for the actual Netflix Prize training set. Use a small integer only for quick debugging.
MAX_TRAIN_ROWS = None

train_df, probe_df, load_seconds = load_train_and_probe_ratings(
    COMBINED_FILES,
    probe_key_set,
    max_train_rows=MAX_TRAIN_ROWS,
)

if len(probe_df) != len(probe_pairs):
    raise ValueError(f"Expected {len(probe_pairs):,} probe ratings but found {len(probe_df):,}.")

print(f"Loaded train rows: {len(train_df):,}")
print(f"Loaded probe rows: {len(probe_df):,}")
print(f"Load seconds: {load_seconds:,.1f}")
train_df.head()

Loaded train rows: 99,072,112
Loaded probe rows: 1,408,395
Load seconds: 72.4


,movie_id,customer_id,rating
0,1,1488844,3.0
1,1,822109,5.0
2,1,885013,4.0
3,1,823519,3.0
4,1,893988,3.0


## Encode IDs for Matrix Factorization

The raw Netflix customer IDs are sparse. We map them to dense row/column indices before fitting the model.

In [5]:
user_index = pd.Index(train_df["customer_id"].unique())
movie_index = pd.Index(train_df["movie_id"].unique())

train_user = user_index.get_indexer(train_df["customer_id"]).astype(np.int32)
train_movie = movie_index.get_indexer(train_df["movie_id"]).astype(np.int32)
train_rating = train_df["rating"].to_numpy(dtype=np.float32)

probe_user = user_index.get_indexer(probe_df["customer_id"]).astype(np.int32)
probe_movie = movie_index.get_indexer(probe_df["movie_id"]).astype(np.int32)
probe_rating = probe_df["rating"].to_numpy(dtype=np.float32)

print(f"Users in training: {len(user_index):,}")
print(f"Movies in training: {len(movie_index):,}")
print(f"Probe rows with unseen user/movie after holdout: {np.sum((probe_user < 0) | (probe_movie < 0)):,}")

Users in training: 480,189
Movies in training: 17,770
Probe rows with unseen user/movie after holdout: 0


## Bias Baseline

A good recommender starts with a baseline: global average + user bias + movie bias. The ALS factors learn the remaining residual structure.

In [6]:
def fit_biases(user_ids, movie_ids, ratings, n_users, n_movies, epochs=8, reg=10.0):
    global_mean = np.float32(ratings.mean())
    user_bias = np.zeros(n_users, dtype=np.float32)
    movie_bias = np.zeros(n_movies, dtype=np.float32)

    user_count = np.bincount(user_ids, minlength=n_users).astype(np.float32)
    movie_count = np.bincount(movie_ids, minlength=n_movies).astype(np.float32)

    for epoch in range(epochs):
        residual_for_users = ratings - global_mean - movie_bias[movie_ids]
        user_sum = np.bincount(user_ids, weights=residual_for_users, minlength=n_users).astype(np.float32)
        user_bias = user_sum / (user_count + reg)

        residual_for_movies = ratings - global_mean - user_bias[user_ids]
        movie_sum = np.bincount(movie_ids, weights=residual_for_movies, minlength=n_movies).astype(np.float32)
        movie_bias = movie_sum / (movie_count + reg)

    return global_mean, user_bias, movie_bias


global_mean, user_bias, movie_bias = fit_biases(
    train_user,
    train_movie,
    train_rating,
    len(user_index),
    len(movie_index),
)

print(f"Global mean rating: {global_mean:.4f}")

Global mean rating: 3.6033


## Explicit ALS Model

This model alternates between solving all user factors while holding movie factors fixed, then solving all movie factors while holding user factors fixed. Each solve is a small ridge-regression problem using NumPy.

In [7]:
def make_grouped_views(primary_ids, secondary_ids, ratings, n_primary):
    order = np.argsort(primary_ids, kind="mergesort")
    primary_sorted = primary_ids[order]
    secondary_sorted = secondary_ids[order]
    rating_sorted = ratings[order]
    counts = np.bincount(primary_sorted, minlength=n_primary)
    offsets = np.empty(n_primary + 1, dtype=np.int64)
    offsets[0] = 0
    np.cumsum(counts, out=offsets[1:])
    return secondary_sorted, rating_sorted, offsets


def solve_factor_block(target_factors, other_factors, other_ids, residual_ratings, offsets, reg):
    rank = other_factors.shape[1]
    eye = np.eye(rank, dtype=np.float32)

    for row_id in range(len(target_factors)):
        start = offsets[row_id]
        stop = offsets[row_id + 1]
        if start == stop:
            continue

        factors = other_factors[other_ids[start:stop]]
        values = residual_ratings[start:stop]
        lhs = factors.T @ factors + reg * eye
        rhs = factors.T @ values
        target_factors[row_id] = np.linalg.solve(lhs, rhs)


def predict_encoded(user_ids, movie_ids, global_mean, user_bias, movie_bias, user_factors=None, movie_factors=None):
    predictions = np.full(len(user_ids), global_mean, dtype=np.float32)

    known_users = user_ids >= 0
    known_movies = movie_ids >= 0
    known_both = known_users & known_movies

    predictions[known_users] += user_bias[user_ids[known_users]]
    predictions[known_movies] += movie_bias[movie_ids[known_movies]]

    if user_factors is not None and movie_factors is not None and np.any(known_both):
        predictions[known_both] += np.sum(
            user_factors[user_ids[known_both]] * movie_factors[movie_ids[known_both]],
            axis=1,
        )

    return np.clip(predictions, 1.0, 5.0)


def rmse(actual, predicted):
    return float(np.sqrt(np.mean((actual - predicted) ** 2)))

In [8]:
def fit_explicit_als(
    user_ids,
    movie_ids,
    ratings,
    n_users,
    n_movies,
    global_mean,
    user_bias,
    movie_bias,
    rank=20,
    epochs=5,
    reg=0.15,
):
    user_factors = RNG.normal(0.0, 0.05, size=(n_users, rank)).astype(np.float32)
    movie_factors = RNG.normal(0.0, 0.05, size=(n_movies, rank)).astype(np.float32)

    movie_by_user, rating_by_user, user_offsets = make_grouped_views(user_ids, movie_ids, ratings, n_users)
    user_by_movie, rating_by_movie, movie_offsets = make_grouped_views(movie_ids, user_ids, ratings, n_movies)

    for epoch in range(1, epochs + 1):
        started = perf_counter()

        user_residual = rating_by_user - global_mean - movie_bias[movie_by_user]
        solve_factor_block(user_factors, movie_factors, movie_by_user, user_residual, user_offsets, reg)

        movie_residual = rating_by_movie - global_mean - user_bias[user_by_movie]
        solve_factor_block(movie_factors, user_factors, user_by_movie, movie_residual, movie_offsets, reg)

        train_pred = predict_encoded(
            user_ids,
            movie_ids,
            global_mean,
            user_bias,
            movie_bias,
            user_factors,
            movie_factors,
        )
        print(f"epoch {epoch}: train RMSE={rmse(ratings, train_pred):.5f}, seconds={perf_counter() - started:,.1f}")

    return user_factors, movie_factors


baseline_probe_pred = predict_encoded(probe_user, probe_movie, global_mean, user_bias, movie_bias)
print(f"Bias-only probe RMSE: {rmse(probe_rating, baseline_probe_pred):.5f}")

user_factors, movie_factors = fit_explicit_als(
    train_user,
    train_movie,
    train_rating,
    len(user_index),
    len(movie_index),
    global_mean,
    user_bias,
    movie_bias,
    rank=20,
    epochs=5,
    reg=0.15,
)

Bias-only probe RMSE: 0.98242
epoch 1: train RMSE=0.87257, seconds=155.7
epoch 2: train RMSE=0.85276, seconds=140.8
epoch 3: train RMSE=0.84400, seconds=174.5
epoch 4: train RMSE=0.83690, seconds=158.9
epoch 5: train RMSE=0.83412, seconds=172.7


## Evaluate on the Actual Netflix Probe Dataset

In [9]:
probe_pred = predict_encoded(
    probe_user,
    probe_movie,
    global_mean,
    user_bias,
    movie_bias,
    user_factors,
    movie_factors,
)

probe_mae = float(np.mean(np.abs(probe_rating - probe_pred)))
probe_rmse = rmse(probe_rating, probe_pred)

print(f"Probe rows evaluated: {len(probe_df):,}")
print(f"Probe RMSE: {probe_rmse:.5f}")
print(f"Probe MAE: {probe_mae:.5f}")

results = probe_df.copy()
results["prediction"] = probe_pred
results.head(10)

Probe rows evaluated: 1,408,395
Probe RMSE: 1.01405
Probe MAE: 0.75863


,movie_id,customer_id,rating,prediction
0,1,30878,4.0,3.806696
1,1,2647871,4.0,3.770682
2,1,1283744,3.0,4.291994
3,1,2488120,5.0,4.504111
4,1,317050,5.0,5.000000
5,1,1904905,4.0,4.237341
6,1,1989766,4.0,3.449207
7,1,14756,4.0,4.288031
8,1,1027056,3.0,3.723824
9,1,1149588,4.0,3.441698


In [10]:
output_path = DATA_DIR / "probe_predictions_numpy_als.csv"
results.to_csv(output_path, index=False)
print(f"Wrote predictions to {output_path}")

Wrote predictions to ../data/netflix/probe_predictions_numpy_als.csv
